In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl

from colliderml_explore.io import load_flat
from colliderml_explore.derived import with_particle_kinematics, with_hit_counts, with_track_kinematics
from colliderml_explore import plots
plots.setup_style()

In [ ]:
flat = load_flat("ttbar", "pu0", max_events=500)
particles = with_particle_kinematics(flat["particles"])
particles = with_hit_counts(particles, flat["tracker_hits"])
hits = flat["tracker_hits"]
tracks = with_track_kinematics(flat["tracks"])

print(f"events: {particles['event_id'].n_unique()}")
print(f"particles: {particles.height:,}, hits: {hits.height:,}, tracks: {tracks.height:,}")
print("\nparticles cols:", particles.columns)
print("\nhits cols:", hits.columns)
print("\ntracks cols:", tracks.columns)

In [ ]:
primary = particles.filter(pl.col("primary"))
primary_charged = primary.filter(pl.col("charge") != 0)

fig, ax = plt.subplots(figsize=(9, 5))
plots.hist_pdg_composition(primary, ax=ax)
ax.set_title("Primary particles by species (ttbar, pu0)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plots.hist_pt(primary, ax=axes[0], label="all primary")
plots.hist_pt(primary_charged, ax=axes[0], label="primary charged")
axes[0].set_xlim(0, 20)
axes[0].set_title("pT spectrum")

plots.hist_eta(primary, ax=axes[1], label="all primary", eta_max = 10)
plots.hist_eta(primary_charged, ax=axes[1], label="primary charged", eta_max = 10)
axes[1].set_title("η distribution")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plots.hist2d_pt_eta(primary_charged, ax=ax)
ax.set_title("Primary charged particles: pT vs η")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plots.hist_num_tracker_hits(primary_charged, ax=ax, label="primary charged")
ax.set_title("Tracker hits per primary charged particle")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plots.hist_impact_param(primary, ax=axes[0], kind="d0",
                        label="primary", range_=(-2, 2))
plots.hist_impact_param(particles.filter(~pl.col("primary")),
                        ax=axes[0], kind="d0",
                        label="secondary", range_=(-2, 2))
axes[0].set_title("Transverse impact parameter d0")

plots.hist_impact_param(primary, ax=axes[1], kind="z0",
                        label="primary", range_=(-10, 10))
plots.hist_impact_param(particles.filter(~pl.col("primary")),
                        ax=axes[1], kind="z0",
                        label="secondary", range_=(-10, 10))
axes[1].set_title("Longitudinal impact parameter z0")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plots.hist_hits_per_event(hits, ax=ax, label="ttbar pu0")
ax.set_title("Tracker hits per event")

In [ ]:
flat_pu200 = load_flat("ttbar", "pu200", max_events=100)
plots.hist_hits_per_event(flat_pu200["tracker_hits"], ax=ax, label="ttbar pu200")
ax.set_xlim(0, 1_000_000)
ax.legend()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plots.hist_residual(hits, ax=axes[0], axis="x", range_=(-0.1, 0.1))
plots.hist_residual(hits, ax=axes[1], axis="y", range_=(-0.1, 0.1))
plots.hist_residual(hits, ax=axes[2], axis="z", range_=(-2, 2))
for a in axes: a.axvline(0, color="grey", linewidth=0.5)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plots.hist_pt(primary_charged, ax=axes[0], label="truth (primary charged)")
plots.hist_pt(tracks, ax=axes[0], label="reco tracks")
axes[0].set_title("pT: truth vs reco")
axes[0].set_xlim(0, 20)

plots.hist_eta(primary_charged, ax=axes[1], label="truth (primary charged)")
plots.hist_eta(tracks, ax=axes[1], label="reco tracks")
axes[1].set_title("η: truth vs reco")
plt.tight_layout()

In [ ]:
ev_id = int(hits["event_id"].unique().sort()[0])

fig = plots.scatter3d_hits(hits, ev_id, color_by="volume_id")
fig.show()

In [ ]:
fig = plots.scatter3d_hits_by_particle(hits, ev_id, top_n_particles=10)
fig.show()

In [ ]:
fig = plots.scatter3d_rho_z_phi(hits, ev_id, color_by="volume_id")
fig.show()